# 1.0 Exploratory Data Analysis: System Log Telemetry & Sequential Patterns

## 1. Business Problem & Solution Framing
Distributed computing clusters (such as Hadoop HDFS, Kubernetes, OpenStack, and microservices) generate terabytes of unstructured console logs daily. 
Traditional monitoring relies on:
- **Static threshold alarms** (e.g., error count > 100/min), which cause high alert fatigue.
- **Keyword/Regex matching** (e.g., searching for `ERROR` or `Exception`), which misses subtle operational sequence corruptions, deadlocks, and silent data corruptions.

### DeepLog Solution:
We treat system execution logs as a natural language sequence. Each log event represents an execution state in the system's runtime grammar. By modeling normal state transitions using Long Short-Term Memory (LSTM) networks, we can detect out-of-order execution, missing steps, and anomalous transitions with near-zero false alarms.

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJ_ROOT = Path("..").resolve()
if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    RAW_LOG_FILE,
    LABEL_FILE,
    PARSED_EVENTS_FILE,
    SESSION_SEQUENCES_FILE,
    TEMPLATES_FILE,
)

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"font.size": 11, "figure.figsize": (10, 5)})
print("Environment and dependencies initialized successfully.")

## 2. Inspecting Raw Telemetry Logs
Raw logs follow the standard HDFS log format:
`Date Time Pid Level Component: Message`

In [ ]:
with open(RAW_LOG_FILE, "r", encoding="utf-8") as f:
    sample_lines = [f.readline().strip() for _ in range(8)]

print(f"Sample raw logs from {RAW_LOG_FILE.name}:\n")
for i, line in enumerate(sample_lines, 1):
    print(f"[{i}] {line}")

## 3. Loading Structured Event Data & Templates
The log parser converts unstructured messages into discrete Event IDs (`E1`, `E2`, etc.) and canonical templates using regular expressions and semantic parsing.

In [ ]:
df_events = pd.read_csv(PARSED_EVENTS_FILE)
with open(TEMPLATES_FILE, "r", encoding="utf-8") as f:
    templates_data = json.load(f)

print(f"Total Parsed Log Events: {len(df_events):,}")
print(f"Unique Log Event Types: {df_events['EventId'].nunique()}")
print(f"Unique Block Sessions:   {df_events['BlockId'].nunique()}")

df_templates = pd.DataFrame(templates_data)[["EventId", "Template", "Component", "Level", "Type"]]
df_templates.head(10)

## 4. Distribution of Log Levels and System Components

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Log level distribution
level_counts = df_events["Level"].value_counts()
colors = ["#2ca02c" if lvl == "INFO" else "#ff7f0e" if lvl == "WARN" else "#d62728" for lvl in level_counts.index]
ax1.bar(level_counts.index, level_counts.values, color=colors, edgecolor="black", alpha=0.85)
ax1.set_title("Log Level Frequency Distribution")
ax1.set_xlabel("Log Severity Level")
ax1.set_ylabel("Log Message Count")
for i, v in enumerate(level_counts.values):
    ax1.text(i, v + 200, f"{v:,}", ha="center", fontweight="bold")

# System component distribution
comp_counts = df_events["Component"].value_counts().head(7)
ax2.barh(comp_counts.index[::-1], comp_counts.values[::-1], color="#1f77b4", edgecolor="black", alpha=0.85)
ax2.set_title("Top System Components by Volume")
ax2.set_xlabel("Event Count")

plt.tight_layout()
plt.show()

## 5. Session Lengths & Anomaly Class Balance
We group logs by their session identifier (`BlockId`) to inspect sequence length distributions and class balance.

In [ ]:
with open(SESSION_SEQUENCES_FILE, "r", encoding="utf-8") as f:
    sessions = json.load(f)

normal_lengths = [s["length"] for s in sessions.values() if s["label"] == "Normal"]
anomaly_lengths = [s["length"] for s in sessions.values() if s["label"] == "Anomaly"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Class balance
labels_count = pd.Series([s["label"] for s in sessions.values()]).value_counts()
ax1.pie(
    labels_count.values,
    labels=labels_count.index,
    autopct="%1.1f%%",
    startangle=140,
    colors=["#2ca02c", "#d62728"],
    explode=(0, 0.08),
    wedgeprops={"edgecolor": "black", "linewidth": 1.2},
)
ax1.set_title(f"Session Class Balance (Total = {len(sessions):,})")

# Sequence length distribution
sns.histplot(normal_lengths, bins=25, color="#2ca02c", label="Normal", kde=True, ax=ax2, alpha=0.6)
sns.histplot(anomaly_lengths, bins=25, color="#d62728", label="Anomaly", kde=True, ax=ax2, alpha=0.6)
ax2.set_title("Event Sequence Length Distribution by Class")
ax2.set_xlabel("Sequence Length (Events per Session)")
ax2.set_ylabel("Session Density")
ax2.legend()

plt.tight_layout()
plt.show()

## 6. Event Transition Matrix (First-Order Sequential Dependencies)
To see why sequential modeling is essential, we compute empirical transition probabilities $P(e_{t+1} \mid e_t)$ across normal execution traces.

In [ ]:
normal_sequences = [s["events"] for s in sessions.values() if s["label"] == "Normal"]

transitions = {}
for seq in normal_sequences:
    for e1, e2 in zip(seq[:-1], seq[1:]):
        if e1 not in transitions:
            transitions[e1] = {}
        transitions[e1][e2] = transitions[e1].get(e2, 0) + 1

# Convert to DataFrame
all_events = sorted(list(set(df_events["EventId"].unique())))
trans_df = pd.DataFrame(0.0, index=all_events, columns=all_events)

for e1, next_dict in transitions.items():
    total = sum(next_dict.values())
    for e2, count in next_dict.items():
        if e1 in trans_df.index and e2 in trans_df.columns:
            trans_df.loc[e1, e2] = count / total

# Plot top events transition heatmap
top_ev = df_events["EventId"].value_counts().head(10).index.tolist()
sub_trans = trans_df.loc[top_ev, top_ev]

plt.figure(figsize=(9, 7))
sns.heatmap(sub_trans, annot=True, fmt=".2f", cmap="YlGnBu", cbar=True, square=True)
plt.title("Empirical First-Order Transition Matrix $P(e_{t+1} \mid e_t)$ (Normal Traces)")
plt.xlabel("Next Event ($e_{t+1}$)")
plt.ylabel("Current Event ($e_t$)")
plt.tight_layout()
plt.show()

## 7. Key Findings & Architecture Motivation
1. **Deterministic Execution Grammar:** Healthy distributed system tasks follow predictable state pathways (e.g. Allocation $\to$ Receive $\to$ AddStoredBlock $\to$ Verify $\to$ Serve).
2. **Subtle Out-of-Order Anomalies:** Anomalies rarely manifest as simple keywords; rather, they present as illegal transitions, premature terminations, or unexpected errors out of sequence.
3. **Need for Long-Range Context:** Simple bigram Markov chains only look back 1 step. An LSTM network with a sliding history window ($h=10$) captures multi-step operational context, enabling precise next-event probability estimation.